# Turkish Hazelnut — Parametric Insurance Pricing

**Objective**: Price event-triggered insurance products for Turkish hazelnut supply shocks.

**Methodology**: Parametric (index-based) pricing rather than OLS regression.
OLS tries to explain *all* year-to-year variance in production, most of which is driven by secular trends,
biennial bearing cycles, and TMO policy — not weather. Frost is a **rare event driver**: it causes large
shocks in the 5–7 years it occurs but cannot explain the other 28 quiet years. The correct frame is:

```
Fair annual premium = P(trigger fires) × E[payout | trigger fires]
Loaded premium      = Fair premium × loading_factor
```

**Perils analysed**
| Peril | Trigger index | Data depth | Signal quality |
|---|---|---|---|
| Spring frost | `april_dh` — degree-hours below −1.5 °C, Apr 1–30 | ERA5 1990–2024 (35 yr) | **Strong** — clear negative production anomaly |
| Summer drought | `spei03` — SPEI-03 at August | ERA5 1950–2024 (75 yr) | **Weak** — signal only at severe levels (n=2) |
| Summer hail | `hail_cp_max` — max monthly conv. precip Jun–Aug | ERA5 1990–2024 (35 yr) | **Negligible** — ERA5 proxy too coarse; basis risk too high |

**Lira depreciation** is excluded from the insurance products — it is already captured in the tradeable basket
(`tryusd`, `tur_usd`, `ulker_try`) and can be hedged with standard FX derivatives.

---
**Note on ERA5 frost depth**: ERA5 covers 1940–present; current frost file covers 1990–2024 (35 years).
Extending to 1940 would give 85 years and tighten frequency estimates significantly.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from pathlib import Path

pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

DATA = Path('../data/raw')

# ── User-adjustable parameters ────────────────────────────────────────────────
LOADING           = 1.35   # premium loading factor (1.35 = 35% above actuarially fair)
SUPPLY_ELASTICITY = -0.50  # hazelnut supply price elasticity (literature: -0.3 to -0.8)
AVG_PRICE_USD_KG  = 3.06   # reference hazelnut price (USD/kg in-shell, 2000–2022 mean)

# Recommended trigger thresholds (calibrated below — change here to re-run all pricing)
FROST_THR_DH      = 3.0    # april_dh (°C·h) — ~1-in-6-year event
DROUGHT_THR_SPEI  = -1.25  # SPEI-03 Aug — severe drought; ~1-in-18-year event

print('Parameters loaded.')
print(f'  Loading factor:        {LOADING:.2f}x')
print(f'  Supply elasticity:     {SUPPLY_ELASTICITY:.2f}')
print(f'  Reference price:       ${AVG_PRICE_USD_KG:.2f}/kg')
print(f'  Frost threshold:       april_dh > {FROST_THR_DH} °C·h')
print(f'  Drought threshold:     SPEI-03 < {DROUGHT_THR_SPEI}')

## 1. Data Load & Production Detrending

Production has a long-run upward trend (tree density, investment). Removing this trend isolates
weather-driven anomalies from structural growth.

In [ ]:
# ── Load all series ───────────────────────────────────────────────────────────
frost = pd.read_csv(DATA / 'era5_frost_monthly.csv', index_col='year')[['march_dh', 'april_dh']]
spei  = pd.read_csv(DATA / 'spei/spei03_era5.csv', index_col='year').rename(columns={'spei03': 'spei03'})
hail_raw = pd.read_csv(DATA / 'era5_hail_monthly.csv', index_col='year')
hail_raw['hail_cp_max'] = hail_raw[['jun_cp_mm', 'jul_cp_mm', 'aug_cp_mm']].max(axis=1)

prod  = pd.read_csv(DATA / 'faostat/turkey_hazelnut_production.csv',
                    index_col='year')['production_mt'].rename('prod_mt')

price_m = pd.read_csv(DATA / 'giresun_spot_prices_monthly.csv')
price_m = price_m[~((price_m['month'] == 8) & (price_m['period'] == 'open'))]
price   = price_m.groupby('crop_year')['avg_usd_kg_inshell'].mean().rename('price_usd_kg')

# ── Build master frame ────────────────────────────────────────────────────────
df = (frost
      .join(hail_raw[['hail_cp_max']])
      .join(spei)
      .join(prod)
      .join(price))
df.index.name = 'year'

# ── Detrend production (linear trend over full FAO history) ──────────────────
prod_full = prod.dropna()
trend_coef = np.polyfit(prod_full.index.astype(float), prod_full.values, 1)
df['prod_trend']    = np.polyval(trend_coef, df.index.astype(float))
df['prod_anom_mt']  = df['prod_mt'] - df['prod_trend']
df['prod_anom_pct'] = df['prod_anom_mt'] / df['prod_trend']

AVG_PROD_MT = df['prod_mt'].mean()

print(f'ERA5 trigger data: {df.dropna(subset=["april_dh"]).index.min()}–{df.dropna(subset=["april_dh"]).index.max()}')
print(f'Production data:   {prod_full.index.min()}–{prod_full.index.max()}, n={len(prod_full)}')
print(f'Price data:        {price.index.min()}–{price.index.max()}, n={price.notna().sum()}')
print(f'Average production (1990–2024): {AVG_PROD_MT/1e3:.0f}k MT')
print(f'Average price (2000–2022): ${df["price_usd_kg"].mean():.2f}/kg')
print(f'\nProduction trend: {trend_coef[0]/1e3:+.1f}k MT/year')

# Plot detrended production
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
s = df['prod_mt'].dropna()
ax1.plot(s.index, s/1e3, 'o-', color='steelblue', lw=1.5, ms=4, label='Actual')
ax1.plot(df.index, df['prod_trend']/1e3, '--', color='black', lw=1.2, label='Linear trend')
ax1.set_ylabel('Production (000 MT)'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('Turkish Hazelnut Production — Actual vs Trend')

anom = df['prod_anom_mt'].dropna()
colors = ['firebrick' if v < 0 else 'steelblue' for v in anom]
ax2.bar(anom.index, anom/1e3, color=colors, alpha=0.8)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_ylabel('Anomaly (000 MT)'); ax2.grid(alpha=0.3)
ax2.set_title('Detrended Production Anomaly  (red = below trend)')

plt.tight_layout(); plt.show()

## 2. Frost — Spring Event Trigger

**Physical mechanism**: Hazelnut buds emerge in late March; young leaves and developing nuts are
vulnerable to sub-zero temperatures in April. Temperatures below −1.5 °C for sustained periods
kill flower primordia and young nutlets, directly reducing yield count.

**Index**: `april_dh` = cumulative degree-hours below −1.5 °C, April 1–30, production-weighted
across Ordu/Giresun/Trabzon/Samsun provinces.

**Trigger structure**: Binary — fires when `april_dh ≥ threshold`. Fixed payout on trigger.

**Data note**: ERA5 frost record currently covers 1990–2024 (35 years). Extending to 1940
would give 85 years for frequency estimation — the single highest-leverage data improvement available.

In [ ]:
# ── Frost threshold calibration ───────────────────────────────────────────────
frost_df = df[['april_dh', 'prod_mt', 'prod_anom_mt', 'prod_anom_pct', 'price_usd_kg']].copy()
frost_df = frost_df[frost_df['april_dh'].notna()]

N_TOTAL_FROST = frost_df['prod_anom_pct'].notna().sum()

print(f'Frost record: {frost_df.index.min()}–{frost_df.index.max()}, n={len(frost_df)}')
print(f'april_dh statistics:')
print(frost_df['april_dh'].describe().round(2).to_string())
print()
print('Threshold sweep — april_dh:')
print(f'{"Threshold":>11} {"P(event)":>9} {"n":>4} {"Event years":>40} {"Mean prod anomaly":>18} {"% below trend":>14}')
print('─' * 110)

sweep_rows = []
for thr in [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]:
    ev = frost_df[frost_df['april_dh'] >= thr]
    anom = ev['prod_anom_pct'].dropna()
    n_ev = len(ev)
    if n_ev < 2:
        continue
    p_ev        = n_ev / len(frost_df)
    pct_below   = (anom < 0).mean()
    mean_anom   = anom.mean()
    event_years = ', '.join(str(y) for y in sorted(ev.index))
    print(f'{thr:>11.1f} {p_ev:>9.1%} {n_ev:>4}   {event_years:<40} {mean_anom:>17.1%} {pct_below:>14.0%}')
    sweep_rows.append(dict(threshold=thr, p_event=p_ev, n_events=n_ev,
                            mean_anom_pct=mean_anom, pct_below_trend=pct_below))

frost_sweep = pd.DataFrame(sweep_rows).set_index('threshold')
print(f'\nRecommended threshold: april_dh ≥ {FROST_THR_DH} °C·h')

In [ ]:
# ── Frost event identification & distribution ─────────────────────────────────
frost_ev   = frost_df['april_dh'] >= FROST_THR_DH
frost_yrs  = frost_df[frost_ev].index.tolist()
clean_yrs  = frost_df[~frost_ev].index.tolist()

ev_anom   = frost_df.loc[frost_df['april_dh'] >= FROST_THR_DH, 'prod_anom_pct'].dropna()
noev_anom = frost_df.loc[frost_df['april_dh'] <  FROST_THR_DH, 'prod_anom_pct'].dropna()

# Mann-Whitney U test — are frost-year anomalies lower?
mw_stat, mw_p = stats.mannwhitneyu(ev_anom, noev_anom, alternative='less')

print(f'Frost event years (april_dh ≥ {FROST_THR_DH}): {frost_yrs}')
print(f'n_frost = {len(ev_anom)},  n_clean = {len(noev_anom)}')
print(f'Mean prod anomaly — frost years: {ev_anom.mean():+.1%}')
print(f'Mean prod anomaly — clean years: {noev_anom.mean():+.1%}')
print(f'Mann-Whitney U test (frost < clean): stat={mw_stat:.1f}, p={mw_p:.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: production anomaly by year, highlighting frost events
ax = axes[0]
s = frost_df['prod_anom_pct'].dropna()
colors = ['firebrick' if y in frost_yrs else ('steelblue' if v >= 0 else 'lightsteelblue')
          for y, v in s.items()]
ax.bar(s.index, s.values * 100, color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
for y in frost_yrs:
    if y in s.index:
        ax.annotate(str(y), xy=(y, s[y]*100), xytext=(0, 6),
                    textcoords='offset points', ha='center', fontsize=7.5, color='firebrick')
ax.set_ylabel('Production anomaly vs trend (%)'); ax.grid(alpha=0.3)
ax.set_title(f'Production anomaly — frost events (red, april_dh ≥ {FROST_THR_DH} DH)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())

# Right: distribution comparison
ax = axes[1]
bins = np.linspace(-0.55, 0.55, 20)
ax.hist(noev_anom * 100, bins=bins * 100, alpha=0.6, color='steelblue', label=f'Clean years (n={len(noev_anom)})')
ax.hist(ev_anom * 100,   bins=bins * 100, alpha=0.8, color='firebrick', label=f'Frost years (n={len(ev_anom)})')
ax.axvline(ev_anom.mean() * 100, color='firebrick', lw=2, ls='--',
           label=f'Frost mean: {ev_anom.mean():+.1%}')
ax.axvline(noev_anom.mean() * 100, color='steelblue', lw=2, ls='--',
           label=f'Clean mean: {noev_anom.mean():+.1%}')
ax.set_xlabel('Production anomaly vs trend (%)'); ax.set_ylabel('Count')
ax.set_title(f'Distribution: frost vs clean years  (MW p={mw_p:.3f})')
ax.legend(fontsize=8.5); ax.grid(alpha=0.3)

plt.suptitle(f'Frost Event Analysis — april_dh ≥ {FROST_THR_DH} °C·h', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Frost premium pricing ─────────────────────────────────────────────────────
# Payout structure: fixed payout L fires when april_dh ≥ FROST_THR_DH
# Expected production shortfall → price spike via supply elasticity

p_frost   = len(ev_anom) / len(frost_df)
mean_prod_loss_pct = abs(ev_anom[ev_anom < 0].mean())  # mean loss in loss-years only
mean_prod_loss_pct_all = abs(ev_anom.mean()) if ev_anom.mean() < 0 else ev_anom.mean()

# Price impact: supply reduction → price spike (using elasticity)
# Δprice% = Δprod% / |elasticity|
implied_price_spike_pct = abs(ev_anom.mean()) / abs(SUPPLY_ELASTICITY)
implied_price_spike_usd = implied_price_spike_pct * AVG_PRICE_USD_KG

fair_rate   = p_frost
loaded_rate = p_frost * LOADING

print('══ FROST PRODUCT PRICING ══')
print(f'Trigger:            april_dh ≥ {FROST_THR_DH} °C·h')
print(f'Historical P(trigger): {p_frost:.1%}  (~1-in-{1/p_frost:.0f} years)')
print(f'Mean prod anomaly (all frost yrs): {ev_anom.mean():+.1%}')
print(f'Mean prod anomaly (loss yrs only): {-mean_prod_loss_pct:.1%}')
print(f'Implied price spike:  +{implied_price_spike_pct:.1%}  = +${implied_price_spike_usd:.2f}/kg')
print()
print(f'Fair premium rate:    {fair_rate:.1%} of payout')
print(f'Loaded premium rate:  {loaded_rate:.1%} of payout  (loading = {LOADING:.2f}x)')
print()

# ── Sensitivity table: payout amount → annual premium ────────────────────────
print('Annual premium by payout size:')
print(f'{"Payout (USD)":>20} {"Fair premium":>16} {"Loaded premium":>16} {"Premium as % of payout":>22}')
print('─' * 80)
for payout in [50_000, 100_000, 250_000, 500_000, 1_000_000, 2_500_000, 5_000_000]:
    fair   = payout * fair_rate
    loaded = payout * loaded_rate
    print(f'{payout:>20,.0f} {fair:>16,.0f} {loaded:>16,.0f} {loaded_rate:>22.1%}')

print()
print('── Buyer reference (10,000 MT annual purchase at $3.06/kg = $30.6M spend) ──')
buyer_vol_mt   = 10_000
expected_excess = buyer_vol_mt * 1000 * implied_price_spike_usd * p_frost  # USD
cover_payout   = buyer_vol_mt * 1000 * implied_price_spike_usd             # full price spike cover
print(f'  Full cover payout:     ${cover_payout:,.0f}')
print(f'  Loaded annual premium: ${cover_payout * loaded_rate:,.0f}')
print(f'  Premium as % of spend: {cover_payout * loaded_rate / (buyer_vol_mt * 1000 * AVG_PRICE_USD_KG):.2%}')

print()
print('── Farmer reference (10 ha, yield 1,800 kg/ha, revenue $54k/year) ──')
farm_ha      = 10
yield_kg_ha  = 1_800
farm_rev     = farm_ha * yield_kg_ha * AVG_PRICE_USD_KG
farm_loss    = farm_rev * abs(ev_anom.mean())
print(f'  Normal revenue:        ${farm_rev:,.0f}')
print(f'  Expected frost loss:   ${farm_loss:,.0f}')
print(f'  Loaded annual premium: ${farm_loss * loaded_rate:,.0f}')
print(f'  Premium as % of revenue: {farm_loss * loaded_rate / farm_rev:.2%}')

In [ ]:
# ── Frost premium sensitivity chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: P(trigger) vs threshold
thresholds = frost_sweep.index
ax = axes[0]
ax.bar(range(len(thresholds)), frost_sweep['p_event'] * 100,
       color=['firebrick' if t == FROST_THR_DH else 'steelblue' for t in thresholds],
       alpha=0.8)
ax.set_xticks(range(len(thresholds)))
ax.set_xticklabels([f'{t} DH' for t in thresholds], rotation=30)
ax.set_ylabel('P(trigger fires) (%)')
ax.set_title('Frost trigger frequency by threshold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)

# Right: loaded premium rate vs payout, at recommended threshold
ax = axes[1]
payouts   = np.logspace(4, 7, 100)
premiums_loaded = payouts * loaded_rate
ax.plot(payouts / 1e6, premiums_loaded / 1e3, color='firebrick', lw=2)
ax.set_xlabel('Payout amount ($M)'); ax.set_ylabel('Annual premium ($k)')
ax.set_title(f'Frost — loaded premium vs payout\n(april_dh ≥ {FROST_THR_DH} DH, rate = {loaded_rate:.1%})')
ax.grid(alpha=0.3)
# Mark buyer and farmer examples
ax.scatter([cover_payout / 1e6], [cover_payout * loaded_rate / 1e3],
           color='navy', zorder=5, s=80, label='Buyer (10k MT)')
ax.scatter([farm_loss / 1e6], [farm_loss * loaded_rate / 1e3],
           color='darkgreen', zorder=5, s=80, label='Farmer (10 ha)')
ax.legend(fontsize=9)

plt.suptitle('Frost Product — Pricing Sensitivity', fontsize=11)
plt.tight_layout(); plt.show()

## 3. Drought — SPEI Trigger

**Physical mechanism**: Moisture deficit in June–August reduces nut fill rate, lowers kernel weight,
and in severe cases causes premature nut drop. Hazelnut trees are relatively drought-tolerant
(deep root systems on the humid Black Sea slopes), so mild drought has limited yield impact.

**Index**: `spei03` — SPEI-03 (3-month Standardised Precipitation-Evapotranspiration Index) at August.
Negative = drier than normal. Scale: −1 = moderate, −1.5 = severe, −2 = extreme drought.

**⚠ Data caveat**: The empirical signal is weak at moderate levels.
Years with SPEI < −1.0 (22.9% frequency) actually show slightly *above-trend* production on average —
suggesting hazelnut is genuinely drought-tolerant at moderate stress. Only severe drought
(SPEI < −1.25, n=2) shows a negative production anomaly, but with only 2 events the
frequency estimate is unreliable (±5–10 pp confidence interval).

**Recommendation**: Price this product with wide uncertainty bounds. Consider SPEI-06 (6-month)
or spring SPEI (April–June) as an alternative index before finalising.

In [ ]:
# ── Drought threshold calibration ─────────────────────────────────────────────
drought_df = df[['spei03', 'prod_anom_mt', 'prod_anom_pct', 'price_usd_kg']].copy()
drought_df = drought_df[drought_df['spei03'].notna()]

print(f'SPEI record: {drought_df.index.min()}–{drought_df.index.max()}, n={len(drought_df)}')
print(f'spei03 statistics:')
print(drought_df['spei03'].describe().round(3).to_string())
print()
print('Threshold sweep — SPEI-03 (trigger fires when SPEI ≤ threshold):')
print(f'{"Threshold":>11} {"P(event)":>9} {"n":>4} {"Event years":>40} {"Mean prod anomaly":>18} {"% below trend":>14}')
print('─' * 110)

drought_sweep_rows = []
for thr in [-0.50, -0.75, -1.00, -1.25, -1.50, -1.75]:
    ev   = drought_df[drought_df['spei03'] <= thr]
    anom = ev['prod_anom_pct'].dropna()
    if len(anom) < 2:
        continue
    p_ev       = len(ev) / len(drought_df)
    pct_below  = (anom < 0).mean()
    event_years = ', '.join(str(y) for y in sorted(ev.index))
    note = ' ← POSITIVE anomaly (hazelnuts drought-tolerant at this level)' if anom.mean() > 0 else ''
    print(f'{thr:>11.2f} {p_ev:>9.1%} {len(anom):>4}   {event_years:<40} {anom.mean():>17.1%}{note}')
    drought_sweep_rows.append(dict(threshold=thr, p_event=p_ev, n_events=len(anom),
                                   mean_anom_pct=anom.mean(), pct_below_trend=pct_below))

drought_sweep = pd.DataFrame(drought_sweep_rows).set_index('threshold')
print(f'\nRecommended threshold: SPEI ≤ {DROUGHT_THR_SPEI} (severe drought, n={len(drought_df[drought_df["spei03"]<=DROUGHT_THR_SPEI])} events)')

In [ ]:
# ── Drought pricing ───────────────────────────────────────────────────────────
drought_ev   = drought_df[drought_df['spei03'] <= DROUGHT_THR_SPEI]
drought_yrs  = drought_ev.index.tolist()
d_anom       = drought_ev['prod_anom_pct'].dropna()
nd_anom      = drought_df.loc[drought_df['spei03'] > DROUGHT_THR_SPEI, 'prod_anom_pct'].dropna()

p_drought         = len(drought_ev) / len(drought_df)
d_price_spike_pct = abs(d_anom.mean()) / abs(SUPPLY_ELASTICITY) if d_anom.mean() < 0 else 0
d_price_spike_usd = d_price_spike_pct * AVG_PRICE_USD_KG

fair_rate_d   = p_drought
loaded_rate_d = p_drought * LOADING

# 95% CI on P(trigger) using Wilson interval (thin sample)
n_total_d = len(drought_df)
n_ev_d    = len(drought_ev)
z = 1.96
p_hat = n_ev_d / n_total_d
wilson_lo = (p_hat + z**2/(2*n_total_d) - z*np.sqrt(p_hat*(1-p_hat)/n_total_d + z**2/(4*n_total_d**2))) / (1 + z**2/n_total_d)
wilson_hi = (p_hat + z**2/(2*n_total_d) + z*np.sqrt(p_hat*(1-p_hat)/n_total_d + z**2/(4*n_total_d**2))) / (1 + z**2/n_total_d)

print('══ DROUGHT PRODUCT PRICING ══')
print(f'Trigger:              SPEI-03 ≤ {DROUGHT_THR_SPEI} (severe drought)')
print(f'Historical P(trigger): {p_drought:.1%}  (~1-in-{1/p_drought:.0f} years)')
print(f'95% Wilson CI:         [{wilson_lo:.1%}, {wilson_hi:.1%}]  ← wide due to n={n_ev_d} events')
print(f'Mean prod anomaly (drought yrs): {d_anom.mean():+.1%}')
print(f'Implied price spike:   +{d_price_spike_pct:.1%}  = +${d_price_spike_usd:.2f}/kg')
print()
print(f'Fair premium rate:     {fair_rate_d:.1%} of payout')
print(f'Loaded premium rate:   {loaded_rate_d:.1%} of payout  (loading = {LOADING:.2f}x)')
print()
print('⚠  Uncertainty is high (n=2 events). Consider adding 50% data-uncertainty margin')
print(f'   Uncertainty-adjusted loaded rate: {loaded_rate_d * 1.5:.1%}')
print()

# Plot SPEI time series with trigger years highlighted
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
spei_s = drought_df['spei03']
ax.plot(spei_s.index, spei_s.values, color='saddlebrown', lw=1.5, marker='o', ms=3)
ax.axhline(DROUGHT_THR_SPEI, color='firebrick', lw=1.5, ls='--',
           label=f'Trigger threshold ({DROUGHT_THR_SPEI})')
ax.axhline(-1.0, color='darkorange', lw=1, ls=':', alpha=0.7, label='Moderate drought (−1.0)')
ax.axhline(0, color='black', lw=0.6)
for y in drought_yrs:
    ax.axvspan(y - 0.4, y + 0.4, color='firebrick', alpha=0.15)
ax.set_ylabel('SPEI-03 (Aug)'); ax.set_title('SPEI-03 August — trigger events highlighted')
ax.legend(fontsize=8.5); ax.grid(alpha=0.3)

ax = axes[1]
bins = np.linspace(-0.55, 0.55, 18)
ax.hist(nd_anom * 100, bins=bins * 100, alpha=0.6, color='steelblue', label=f'Normal years (n={len(nd_anom)})')
ax.hist(d_anom * 100,  bins=bins * 100, alpha=0.8, color='firebrick',  label=f'Drought years (n={len(d_anom)})')
ax.axvline(d_anom.mean() * 100, color='firebrick', lw=2, ls='--',
           label=f'Drought mean: {d_anom.mean():+.1%}')
ax.set_xlabel('Production anomaly vs trend (%)'); ax.set_ylabel('Count')
ax.set_title(f'Distribution: drought vs normal  (n_drought={len(d_anom)} — USE WITH CAUTION)')
ax.legend(fontsize=8.5); ax.grid(alpha=0.3)

plt.suptitle(f'Drought Event Analysis — SPEI-03 ≤ {DROUGHT_THR_SPEI}', fontsize=11)
plt.tight_layout(); plt.show()

## 4. Hail — ERA5 Proxy Analysis

**Physical mechanism**: Large hailstones in June–August physically damage developing nuts and
leaves, causing direct yield loss. Damage is extremely localised — a storm can hit one
valley and miss the next.

**⚠ Index quality problem**: ERA5 `convective_precipitation` at 0.25° (~28 km) resolution
is a poor proxy for hail. The analysis below shows **no statistically meaningful production signal**
at any threshold — the % of frost years below trend is 14–27%, *below* the 50% baseline.

This does not mean hail doesn't damage hazelnuts — it means ERA5 cannot measure it reliably.

**To build a credible hail product**, better data sources are needed:
- **TSMS station records**: Turkish State Meteorological Service hourly data, includes
  hail obs at ~500 stations. Request via TSMS data portal.
- **EUMETSAT MSG SEVIRI**: Satellite-based hail proxy at ~3 km resolution.
- **ESWD (European Severe Weather Database)**: Reports of damaging hail events in Turkey.

**Conclusion**: Hail product design is **deferred** pending better index data.

In [ ]:
# ── Hail signal assessment ─────────────────────────────────────────────────────
hail_df = df[['hail_cp_max', 'prod_anom_pct']].dropna()

print('Hail proxy (ERA5 max monthly conv. precip, Jun–Aug) — threshold sweep:')
print(f'{"Percentile":>12} {"Threshold":>12} {"P(event)":>9} {"n":>4} '
      f'{"Mean prod anomaly":>18} {"% below trend":>14}  Signal?')
print('─' * 100)
for pct_thr in [65, 70, 75, 80, 85, 90, 95]:
    thr = hail_df['hail_cp_max'].quantile(pct_thr / 100)
    ev   = hail_df[hail_df['hail_cp_max'] >= thr]
    anom = ev['prod_anom_pct'].dropna()
    if len(anom) < 2:
        continue
    p_ev      = len(ev) / len(hail_df)
    pct_below = (anom < 0).mean()
    signal    = '✓ possible' if pct_below >= 0.60 and anom.mean() < -0.03 else '✗ no signal'
    print(f'{pct_thr:>12}th {thr:>10.1f}mm {p_ev:>9.1%} {len(anom):>4} '
          f'{anom.mean():>17.1%} {pct_below:>14.0%}  {signal}')

print()
print('Conclusion: ERA5 convective precipitation does not carry a usable hail damage signal.')
print('Hail product requires station-level or satellite hail data before pricing is possible.')

# Plot anyway for completeness
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(hail_df['hail_cp_max'], hail_df['prod_anom_pct'] * 100,
           color='purple', alpha=0.7, s=50)
m, b, r, p_val, se = stats.linregress(hail_df['hail_cp_max'], hail_df['prod_anom_pct'] * 100)
xfit = np.linspace(hail_df['hail_cp_max'].min(), hail_df['hail_cp_max'].max(), 50)
ax.plot(xfit, m * xfit + b, color='purple', lw=1.5, ls='--',
        label=f'OLS fit (r={np.sqrt(r**2) * np.sign(m):.2f}, p={p_val:.2f})')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Hail proxy — max monthly conv. precip Jun–Aug (mm)')
ax.set_ylabel('Production anomaly vs trend (%)')
ax.set_title('Hail proxy vs production anomaly — no meaningful signal (ERA5 too coarse)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Multi-Peril Correlation

Frost (spring) and drought (summer) operate in different seasons, so they are expected to be
largely independent. If independent, the premium for a combined product is approximately
the sum of the individual premiums (no diversification benefit for the insurer,
no compounding penalty for the buyer).

In [ ]:
# ── Multi-peril correlation and joint probability ─────────────────────────────
overlap_df = df[['april_dh', 'spei03', 'prod_anom_pct']].dropna()
overlap_df['frost_ev']   = overlap_df['april_dh'] >= FROST_THR_DH
overlap_df['drought_ev'] = overlap_df['spei03']   <= DROUGHT_THR_SPEI
overlap_df['both']       = overlap_df['frost_ev'] & overlap_df['drought_ev']

n_total  = len(overlap_df)
n_frost  = overlap_df['frost_ev'].sum()
n_drt    = overlap_df['drought_ev'].sum()
n_both   = overlap_df['both'].sum()

p_f   = n_frost / n_total
p_d   = n_drt   / n_total
p_b   = n_both  / n_total
p_either   = p_f + p_d - p_b
p_indep_b  = p_f * p_d  # expected joint prob if independent

print(f'Overlapping sample: {overlap_df.index.min()}–{overlap_df.index.max()}, n={n_total}')
print()
print(f'P(frost)              = {p_f:.1%}  years: {list(overlap_df[overlap_df["frost_ev"]].index)}')
print(f'P(drought)            = {p_d:.1%}  years: {list(overlap_df[overlap_df["drought_ev"]].index)}')
print(f'P(both)               = {p_b:.1%}  years: {list(overlap_df[overlap_df["both"]].index)}')
print(f'P(both) if independent= {p_indep_b:.1%}  (= P(frost) × P(drought))')
print(f'P(frost OR drought)   = {p_either:.1%}')
print()

# Chi-squared test of independence
ct = pd.crosstab(overlap_df['frost_ev'], overlap_df['drought_ev'])
chi2, chi_p, dof, _ = stats.chi2_contingency(ct)
print(f'Chi-squared independence test: χ²={chi2:.2f}, p={chi_p:.3f}')
print(f'→ {"Cannot reject independence" if chi_p > 0.10 else "Evidence of dependence"} (p={chi_p:.3f})')
print()

# Compound event impact
if n_both > 0:
    both_anom = overlap_df.loc[overlap_df['both'], 'prod_anom_pct'].dropna()
    print(f'Compound event (frost AND drought) production anomaly: {both_anom.mean():+.1%}')
    print(f'  vs frost-only: {overlap_df.loc[overlap_df["frost_ev"] & ~overlap_df["drought_ev"], "prod_anom_pct"].mean():+.1%}')
    print(f'  vs drought-only: {overlap_df.loc[~overlap_df["frost_ev"] & overlap_df["drought_ev"], "prod_anom_pct"].mean():+.1%}')

# Visualise
fig, ax = plt.subplots(figsize=(8, 7))
colors_map = {(False,False):'steelblue', (True,False):'firebrick',
              (False,True):'darkorange', (True,True):'purple'}
labels_map = {(False,False):'Clean', (True,False):'Frost only',
              (False,True):'Drought only', (True,True):'Both'}
for (fe, de), grp in overlap_df.groupby(['frost_ev','drought_ev']):
    ax.scatter(grp['april_dh'], grp['spei03'],
               color=colors_map[(fe,de)], label=labels_map[(fe,de)],
               s=70, alpha=0.85, zorder=3)
    for yr, row in grp.iterrows():
        ax.annotate(str(yr), (row['april_dh'], row['spei03']),
                    xytext=(4,3), textcoords='offset points', fontsize=7.5)
ax.axvline(FROST_THR_DH, color='firebrick', ls='--', lw=1.2, alpha=0.8)
ax.axhline(DROUGHT_THR_SPEI, color='darkorange', ls='--', lw=1.2, alpha=0.8)
ax.set_xlabel('april_dh (°C·h)'); ax.set_ylabel('SPEI-03 August')
ax.set_title('Frost vs Drought — event space\n(dashed lines = trigger thresholds)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Premium Summary & Product Structures

In [ ]:
# ── Full sensitivity: threshold × payout → annual premium ────────────────────
from itertools import product as iproduct

frost_thresholds   = [1.0, 2.0, 3.0, 5.0, 7.0]
drought_thresholds = [-0.75, -1.00, -1.25, -1.50]
payout_sizes       = [100_000, 250_000, 500_000, 1_000_000, 2_500_000]

print('═' * 85)
print('FROST PRODUCT — Loaded annual premium by trigger threshold and payout size')
print('═' * 85)
header = f'{"Threshold":>12}  {"P(event)":>9}' + ''.join(f'  Payout ${p/1e3:.0f}k' for p in payout_sizes)
print(header)
print('─' * 85)
for thr in frost_thresholds:
    ev    = frost_df[frost_df['april_dh'] >= thr]
    p_ev  = len(ev) / len(frost_df)
    rate  = p_ev * LOADING
    premiums = ''.join(f'  ${p * rate / 1e3:>8.1f}k' for p in payout_sizes)
    marker = ' ◄ recommended' if thr == FROST_THR_DH else ''
    print(f'{thr:>9.1f} DH  {p_ev:>9.1%}{premiums}{marker}')

print()
print('═' * 85)
print('DROUGHT PRODUCT — Loaded annual premium by SPEI threshold and payout size')
print('⚠  Thin sample (n=2–8 events) — treat as indicative only')
print('═' * 85)
header = f'{"Threshold":>12}  {"P(event)":>9}' + ''.join(f'  Payout ${p/1e3:.0f}k' for p in payout_sizes)
print(header)
print('─' * 85)
for thr in drought_thresholds:
    ev   = drought_df[drought_df['spei03'] <= thr]
    p_ev = len(ev) / len(drought_df)
    rate = p_ev * LOADING
    premiums = ''.join(f'  ${p * rate / 1e3:>8.1f}k' for p in payout_sizes)
    marker = ' ◄ recommended' if thr == DROUGHT_THR_SPEI else ''
    print(f'{thr:>9.2f}     {p_ev:>9.1%}{premiums}{marker}')

In [ ]:
# ── Product spec summary ──────────────────────────────────────────────────────
frost_p   = len(frost_df[frost_df['april_dh']   >= FROST_THR_DH])   / len(frost_df)
drought_p = len(drought_df[drought_df['spei03'] <= DROUGHT_THR_SPEI]) / len(drought_df)

print('═' * 80)
print('HAZELNUT PARAMETRIC INSURANCE — RECOMMENDED PRODUCT STRUCTURES')
print('═' * 80)

products = [
    {
        'name':       'Frost Event Cover (FEC)',
        'peril':      'Spring frost — young nut damage',
        'index':      'april_dh (ERA5, °C·h below −1.5°C, Apr 1–30)',
        'trigger':    f'april_dh ≥ {FROST_THR_DH} °C·h',
        'p_event':    frost_p,
        'freq':       f'~1-in-{1/frost_p:.0f} years',
        'payout':     'Fixed; buyer: ~14% of annual hazelnut spend / farmer: ~7% of revenue',
        'rate':       frost_p * LOADING,
        'confidence': 'MEDIUM — 35-yr record, 6 events; extend ERA5 to 1940 to upgrade to HIGH',
        'basis_risk': 'Low — ERA5 frost tracks provincial temps reliably',
    },
    {
        'name':       'Severe Drought Cover (SDC)',
        'peril':      'Summer moisture deficit — nut fill loss',
        'index':      'SPEI-03 August (ERA5-derived)',
        'trigger':    f'SPEI-03 ≤ {DROUGHT_THR_SPEI}',
        'p_event':    drought_p,
        'freq':       f'~1-in-{1/drought_p:.0f} years',
        'payout':     'Fixed; sized to cover ~7% revenue/supply shortfall',
        'rate':       drought_p * LOADING * 1.5,  # extra 50% for data uncertainty
        'confidence': 'LOW — only 2 events in 35-yr window; verify with spring SPEI-06',
        'basis_risk': 'Medium — SPEI-03 Aug may not capture the critical June–July window',
    },
]

for p in products:
    print()
    print(f'  ── {p["name"]} ──')
    print(f'  Peril:           {p["peril"]}')
    print(f'  Index:           {p["index"]}')
    print(f'  Trigger:         {p["trigger"]}')
    print(f'  Event frequency: {p["p_event"]:.1%} ({p["freq"]})')
    print(f'  Payout guidance: {p["payout"]}')
    print(f'  Premium rate:    {p["rate"]:.1%} of payout per year')
    print(f'  Data confidence: {p["confidence"]}')
    print(f'  Basis risk:      {p["basis_risk"]}')

print()
print('  ── Hail Cover ──  STATUS: DEFERRED')
print('  ERA5 convective precip shows no production signal. Requires TSMS station')
print('  hail records or EUMETSAT satellite data before pricing is possible.')

print()
print('  ── Combined Cover (Frost + Drought) ──')
p_either_combined = frost_p + drought_p - frost_p * drought_p
rate_combined = p_either_combined * LOADING
print(f'  P(frost OR drought) ≈ {p_either_combined:.1%} (assuming independence)')
print(f'  Combined premium rate: {rate_combined:.1%} of payout per year')
print(f'  (Frost and drought co-occurred in 2003 only — 1 event in 35 years)')

In [ ]:
# ── Summary heatmap: premium rate by peril × threshold ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Frost heatmap
frost_thrs = [0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
payouts_ax = [100_000, 250_000, 500_000, 1_000_000, 2_500_000]

for ax, (thresholds_list, df_src, col, direction, title, cmap) in zip(axes, [
    (frost_thrs, frost_df, 'april_dh', 'ge', 'Frost — annual premium ($k)\n(rows=threshold, cols=payout size)', 'Reds'),
    ([-0.50,-0.75,-1.00,-1.25,-1.50], drought_df, 'spei03', 'le',
     'Drought — annual premium ($k)\n(rows=threshold, cols=payout size)', 'Oranges'),
]):
    matrix = []
    for thr in thresholds_list:
        n_df = df_src[col].notna().sum()
        if direction == 'ge':
            n_ev = (df_src[col] >= thr).sum()
        else:
            n_ev = (df_src[col] <= thr).sum()
        rate = (n_ev / n_df) * LOADING
        matrix.append([p * rate / 1e3 for p in payouts_ax])

    mat = np.array(matrix)
    im = ax.imshow(mat, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(payouts_ax)))
    ax.set_xticklabels([f'${p/1e3:.0f}k' for p in payouts_ax], fontsize=9)
    ax.set_yticks(range(len(thresholds_list)))
    ax.set_yticklabels([str(t) for t in thresholds_list], fontsize=9)
    ax.set_xlabel('Payout size')
    ax.set_ylabel('Trigger threshold')
    ax.set_title(title)
    for i in range(len(thresholds_list)):
        for j in range(len(payouts_ax)):
            ax.text(j, i, f'${mat[i,j]:.1f}k', ha='center', va='center',
                    fontsize=8, color='white' if mat[i,j] > mat.max()*0.6 else 'black')
    plt.colorbar(im, ax=ax, label='Annual premium ($k)')

plt.suptitle(f'Premium Heatmap — Loading {LOADING:.2f}x', fontsize=11)
plt.tight_layout(); plt.show()